# ORQUESTACIÓN DEL PIPELINE

Este notebook:
1. Detecta automáticamente si se ejecuta en **Google Colab** o en **local** (VS Code / Jupyter).
2. Lee la configuración desde `config.xlsx`.
3. Ejecuta en secuencia los notebooks marcados con `ejecutar=True` en la hoja `pipeline`.

In [ ]:
# ============================================================
# 0. DETECCIÓN DE ENTORNO
# ============================================================

import os
import sys
import subprocess
import time

import pandas as pd

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/TFM_NoeliaGarciaGarcia'
else:
    # En local, PROJECT_ROOT es relativo a la carpeta NOTEBOOKS
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
    # Si se ejecuta desde NOTEBOOKS, sube un nivel a Pipeline
    if os.path.basename(PROJECT_ROOT) == 'Pipeline':
        PROJECT_ROOT = PROJECT_ROOT  # Ya estamos en Pipeline
    elif os.path.basename(os.getcwd()) == 'NOTEBOOKS':
        PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))

NOTEBOOKS_DIR = os.path.join(PROJECT_ROOT, 'Pipeline', 'NOTEBOOKS')
if not os.path.isdir(NOTEBOOKS_DIR):
    # Caso: PROJECT_ROOT ya apunta a Pipeline/
    NOTEBOOKS_DIR = os.path.join(PROJECT_ROOT, 'NOTEBOOKS')

CONFIG_PATH = os.path.join(NOTEBOOKS_DIR, 'config.xlsx')

print(f"Entorno: {'Google Colab' if IN_COLAB else 'Local'}")
print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"NOTEBOOKS_DIR: {NOTEBOOKS_DIR}")
print(f"CONFIG_PATH: {CONFIG_PATH}")
print(f"Config existe: {os.path.isfile(CONFIG_PATH)}")

Entorno: Local
PROJECT_ROOT: g:\Mi unidad\TFM_NoeliaGarciaGarcia\Pipeline
NOTEBOOKS_DIR: g:\Mi unidad\TFM_NoeliaGarciaGarcia\Pipeline\NOTEBOOKS
CONFIG_PATH: g:\Mi unidad\TFM_NoeliaGarciaGarcia\Pipeline\NOTEBOOKS\config.xlsx
Config existe: True


In [ ]:
# ============================================================
# 1. LECTURA DE CONFIGURACIÓN
# ============================================================

import pandas as pd

# Leer parámetros generales
df_params = pd.read_excel(CONFIG_PATH, sheet_name='config')
print("=== PARÁMETROS ===")
display(df_params)

# Leer pipeline (qué notebooks ejecutar)
df_pipeline = pd.read_excel(CONFIG_PATH, sheet_name='pipeline')
print("\n=== PIPELINE ===")
display(df_pipeline)

=== PARÁMETROS ===


,parameter,value
0,window_minutes,15
1,fs,8
2,samples_per_window,7200
3,min_k_clusters,2
4,max_k_clusters,8
5,apply_shift,1
6,shift_min_samples,-8
7,shift_max_samples,32
8,k_final,2



=== PIPELINE ===


,notebook,descripcion,ejecutar
0,1.limpieza_ventanas,Limpieza de datos por ventanas,True
1,2.flujo,Calculo de flujo de O2,True
2,3.limpieza_global,Limpieza global de datos agregados,False
3,4.oleaje,Integracion de datos de oleaje,False
4,5.clasificacion,Clasificacion K-Means del oleaje,False
5,6.estudio,Estudio de correlaciones,False
6,7.modelos,Entrenamiento de modelos,False
7,8.comparacion_modelos,Comparacion de modelos,False
8,9.graficas,Graficas y visualizacion,False


In [ ]:
# ============================================================
# 2. EJECUCIÓN SELECTIVA DEL PIPELINE
# ============================================================

notebooks_to_run = df_pipeline[df_pipeline['ejecutar'] == True]['notebook'].tolist()

print(f"Notebooks a ejecutar: {len(notebooks_to_run)}")
for nb in notebooks_to_run:
    print(f"  ✓ {nb}")

notebooks_skipped = df_pipeline[df_pipeline['ejecutar'] == False]['notebook'].tolist()
if notebooks_skipped:
    print(f"\nNotebooks omitidos: {len(notebooks_skipped)}")
    for nb in notebooks_skipped:
        print(f"  ✗ {nb}")

Notebooks a ejecutar: 2
  ✓ 1.limpieza_ventanas
  ✓ 2.flujo

Notebooks omitidos: 7
  ✗ 3.limpieza_global
  ✗ 4.oleaje
  ✗ 5.clasificacion
  ✗ 6.estudio
  ✗ 7.modelos
  ✗ 8.comparacion_modelos
  ✗ 9.graficas


In [14]:
# ============================================================
# 3. EJECUTAR NOTEBOOKS
# ============================================================
#
# Cada notebook se ejecuta con nbconvert como subproceso.
# - En Colab: se usa subprocess + nbconvert.
# - En local: igual, usando el kernel de Python actual.
#
# Si un notebook falla, se muestra el error y se pregunta
# si continuar con el siguiente.
# ============================================================

results = []

for nb_name in notebooks_to_run:
    nb_path = os.path.join(NOTEBOOKS_DIR, f"{nb_name}.ipynb")

    if not os.path.isfile(nb_path):
        print(f"\n⚠ No se encontró: {nb_path}")
        results.append({'notebook': nb_name, 'status': 'NO ENCONTRADO', 'time_s': 0})
        continue

    print(f"\n{'='*60}")
    print(f"Ejecutando: {nb_name}")
    print(f"{'='*60}")

    start = time.time()

    try:
        result = subprocess.run(
            [
                sys.executable, '-m', 'jupyter', 'nbconvert',
                '--to', 'notebook',
                '--execute',
                '--inplace',
                '--ExecutePreprocessor.timeout=600',
                nb_path
            ],
            capture_output=True,
            text=True
        )

        elapsed = time.time() - start

        if result.returncode == 0:
            print(f"✓ Completado en {elapsed:.1f}s")
            results.append({'notebook': nb_name, 'status': 'OK', 'time_s': round(elapsed, 1)})
        else:
            print(f"✗ Error (código {result.returncode})")
            print(result.stderr[-2000:] if len(result.stderr) > 2000 else result.stderr)
            results.append({'notebook': nb_name, 'status': 'ERROR', 'time_s': round(elapsed, 1)})

    except Exception as e:
        elapsed = time.time() - start
        print(f"✗ Excepción: {e}")
        results.append({'notebook': nb_name, 'status': 'EXCEPCIÓN', 'time_s': round(elapsed, 1)})

# Resumen final
print(f"\n{'='*60}")
print("RESUMEN DE EJECUCIÓN")
print(f"{'='*60}")
df_results = pd.DataFrame(results)
display(df_results)


Ejecutando: 1.limpieza_ventanas
✓ Completado en 194.6s

Ejecutando: 2.flujo
✓ Completado en 28.1s

RESUMEN DE EJECUCIÓN


,notebook,status,time_s
0,1.limpieza_ventanas,OK,194.6
1,2.flujo,OK,28.1
